# 🧪 06 · Test Cases at Scale — Batch + Export — Advanced track

**Level:** Experts · **Time:** ~30 min

Notebook 02 generated test cases for *one* requirement. Here we make it **production-shaped**:
- **Validate** the model's JSON and **retry** automatically if it's malformed
- **Batch** a whole backlog of requirements in one run
- **Export** to a **Jira/TestRail-style CSV** and to **Gherkin `.feature` files**

> ⚙️ Enable GPU: *Runtime → Change runtime type → T4 GPU*.

### Step 1 — Setup

In [ ]:
!pip install -q -U transformers accelerate
import torch, json, re, logging, os
from transformers import pipeline

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
log = logging.getLogger('qa-batch')

chat = pipeline('text-generation', model='Qwen/Qwen2.5-1.5B-Instruct',
                torch_dtype=torch.bfloat16, device_map='auto')
print('✅ ready')

### Step 2 — Generator with validation + auto-retry
If the model returns broken JSON or misses required fields, we ask again (up to 3 tries)
before giving up. This is the difference between a demo and a tool you can trust in a pipeline.

In [ ]:
SYSTEM = ('You are a senior QA engineer. For a requirement, return ONLY valid JSON '
          '(no markdown) shaped as: {"test_cases":[{"id":"TC-01","title":"...",'
          '"type":"positive|negative|edge|boundary","priority":"high|medium|low",'
          '"steps":["..."],"expected":"..."}]}')

REQUIRED = {"id", "title", "type", "priority", "steps", "expected"}

def _valid(data):
    tcs = data.get("test_cases")
    return isinstance(tcs, list) and len(tcs) > 0 and all(REQUIRED <= set(tc) for tc in tcs)

def generate(requirement, tries=3):
    for attempt in range(1, tries + 1):
        user = f"Requirement:\n{requirement}\n\nGenerate 6-10 test cases as JSON."
        out = chat([{'role':'system','content':SYSTEM}, {'role':'user','content':user}],
                   max_new_tokens=1200, do_sample=False)
        text = out[0]['generated_text'][-1]['content']
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                data = json.loads(m.group(0))
                if _valid(data):
                    return data["test_cases"]
            except json.JSONDecodeError:
                pass
        log.warning('Attempt %d gave invalid output, retrying...', attempt)
    log.error('Could not get valid JSON after %d tries for: %.60s', tries, requirement)
    return []

print('✅ validated generator ready')

### Step 3 — Your backlog (batch input)
Edit this list, or upload a `.txt` (one requirement per line) and load it instead.

In [ ]:
REQUIREMENTS = [
    "As a user, I can reset my password via an email link that expires in 15 minutes.",
    "As a shopper, I can add up to 10 items to my cart and see the running total update.",
    "As a user, I can upload a profile picture (JPG/PNG only, max 5 MB).",
    "As an admin, I can deactivate a user account, after which that user cannot log in.",
]

# To load from an uploaded file instead, uncomment:
# REQUIREMENTS = [l.strip() for l in open('my_requirements.txt') if l.strip() and not l.startswith('#')]

print(f'{len(REQUIREMENTS)} requirements to process')

### Step 4 — Run the batch

In [ ]:
import pandas as pd

rows = []
for r_idx, req in enumerate(REQUIREMENTS, 1):
    print(f'[{r_idx}/{len(REQUIREMENTS)}] {req[:70]}...')
    for tc in generate(req):
        steps = tc.get('steps', [])
        rows.append({
            'requirement_id': f'REQ-{r_idx:02d}',
            'requirement': req,
            'test_id': tc.get('id', ''),
            'title': tc.get('title', ''),
            'type': tc.get('type', ''),
            'priority': tc.get('priority', ''),
            'steps': ' | '.join(steps) if isinstance(steps, list) else str(steps),
            'expected': tc.get('expected', ''),
        })

df = pd.DataFrame(rows)
print(f'\n✅ generated {len(df)} test cases across {len(REQUIREMENTS)} requirements')
df.head(12)

### Step 5 — Export to CSV (import into Jira / TestRail / a spreadsheet)

In [ ]:
df.to_csv('test_cases_all.csv', index=False)
print('✅ saved test_cases_all.csv  (download from the 📁 Files panel)')

### Step 6 — Export to Gherkin `.feature` files (for BDD / automation)

In [ ]:
os.makedirs('features', exist_ok=True)

def to_gherkin(req_id, requirement, group):
    lines = [f'Feature: {requirement}', '']
    for _, tc in group.iterrows():
        lines.append(f"  Scenario: {tc['title']} ({tc['type']}, {tc['priority']})")
        steps = tc['steps'].split(' | ')
        if steps:
            lines.append(f'    Given {steps[0]}')
            for s in steps[1:]:
                lines.append(f'    And {s}')
        lines.append(f"    Then {tc['expected']}")
        lines.append('')
    return '\n'.join(lines)

for req_id, group in df.groupby('requirement_id'):
    requirement = group['requirement'].iloc[0]
    with open(f'features/{req_id}.feature', 'w') as f:
        f.write(to_gherkin(req_id, requirement, group))

import shutil
shutil.make_archive('qa_features', 'zip', 'features')
print('✅ wrote features/*.feature and qa_features.zip  (download from the 📁 Files panel)')
print(open(f'features/REQ-01.feature').read())

---
### 🧠 Expert discussion
- **Validation + retry** is what makes LLM output safe to put in a pipeline — never trust a single raw response.
- This whole notebook could run as a **scheduled job** that turns new stories into draft test cases automatically (with a human reviewing the output).
- Map the CSV columns to your real test-management tool's import format.
- A human still reviews everything — the AI scales the *drafting*, not the *judgement*.